In [2]:
import pandas as pd
import numpy as np
import os
import warnings

from scipy.signal import savgol_filter

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Load data
# -----------------------------
def load_data():

    print("Loading data...")

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


# -----------------------------
# Prepare spectral features
# -----------------------------
def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    print("Number of spectral features:", X.shape[1])

    return X, y, X_test


# -----------------------------
# Savitzky–Golay smoothing
# -----------------------------
def apply_savgol(X, X_test):

    print("Applying Savitzky–Golay smoothing...")

    X_sg = savgol_filter(
        X,
        window_length=11,
        polyorder=2,
        deriv=0,
        axis=1
    )

    X_test_sg = savgol_filter(
        X_test,
        window_length=11,
        polyorder=2,
        deriv=0,
        axis=1
    )

    return X_sg, X_test_sg


# -----------------------------
# Feature scaling
# -----------------------------
def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    return X_scaled, X_test_scaled


# -----------------------------
# ElasticNet KFold ensemble
# -----------------------------
def elasticnet_kfold_ensemble(X, y, X_test):

    print("\nStarting ElasticNet KFold ensemble...\n")

    kf = KFold(n_splits=8, shuffle=True, random_state=42)

    # Reduced alpha range for stability
    alphas = np.logspace(-3, 0, 20)

    l1_ratio = 0.5

    best_alpha = None
    best_score = np.inf

    print("Searching alpha...\n")

    for alpha in alphas:

        rmse_scores = []

        for train_idx, val_idx in kf.split(X):

            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model = ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=20000,
                tol=1e-3,
                selection="random",
                random_state=42
            )

            model.fit(X_train, y_train)

            preds = model.predict(X_val)

            rmse = np.sqrt(mean_squared_error(y_val, preds))
            rmse_scores.append(rmse)

        mean_rmse = np.mean(rmse_scores)

        print(f"alpha={alpha:.5f}  RMSE={mean_rmse:.4f}")

        if mean_rmse < best_score:
            best_score = mean_rmse
            best_alpha = alpha

    print("\nBest alpha:", best_alpha)
    print("Best CV RMSE:", best_score)

    # -----------------------------
    # Train fold models
    # -----------------------------
    print("\nTraining fold ensemble...\n")

    test_preds = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        X_train = X[train_idx]
        y_train = y[train_idx]

        model = ElasticNet(
            alpha=best_alpha,
            l1_ratio=l1_ratio,
            max_iter=20000,
            tol=1e-3,
            selection="random",
            random_state=42
        )

        model.fit(X_train, y_train)

        fold_preds = model.predict(X_test)

        test_preds += fold_preds / kf.n_splits

        print(f"Fold {fold+1} complete")

    return test_preds


# -----------------------------
# Save submission
# -----------------------------
def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    experiment_name = "exp16_elasticnet_kfold8_savgol_logalpha_fast_20260325"

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("\nSubmission saved:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


# -----------------------------
# Main pipeline
# -----------------------------
def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = apply_savgol(X, X_test)

    X, X_test = scale_features(X, X_test)

    preds = elasticnet_kfold_ensemble(X, y, X_test)

    save_submission(test, preds)


main()

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Number of spectral features: 1555
Applying Savitzky–Golay smoothing...

Starting ElasticNet KFold ensemble...

Searching alpha...

alpha=0.00100  RMSE=13.7926
alpha=0.00144  RMSE=14.2988
alpha=0.00207  RMSE=14.7940
alpha=0.00298  RMSE=15.2878
alpha=0.00428  RMSE=15.7748
alpha=0.00616  RMSE=16.2483
alpha=0.00886  RMSE=16.7085
alpha=0.01274  RMSE=17.1425
alpha=0.01833  RMSE=17.5847
alpha=0.02637  RMSE=18.0465
alpha=0.03793  RMSE=18.5364
alpha=0.05456  RMSE=19.0490
alpha=0.07848  RMSE=19.5711
alpha=0.11288  RMSE=20.0679
alpha=0.16238  RMSE=20.5742
alpha=0.23357  RMSE=21.1422
alpha=0.33598  RMSE=21.7142
alpha=0.48329  RMSE=22.2013
alpha=0.69519  RMSE=22.8526
alpha=1.00000  RMSE=23.7955

Best alpha: 0.001
Best CV RMSE: 13.792610319925698

Training fold ensemble...

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete
Fold 6 complete
Fold 7 complete
Fold 8 complete

Submission saved: ../submissions/e